# Iceberg Basics — 04: Time Travel

## What you will learn
Iceberg time travel lets you query historical data without keeping
separate copies — the snapshot history provides free versioning.

In this notebook:
1. `VERSION AS OF` — query by snapshot ID
2. `TIMESTAMP AS OF` — query by point in time
3. Named references — tags as reusable checkpoints
4. Comparing data between two points in time
5. Use cases — debugging, auditing, ML reproducibility
6. Time travel limitations — retention and VACUUM


In [ ]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

In [ ]:
spark.sql("DROP TABLE IF EXISTS local.icedb.tt_orders")
df.writeTo("local.icedb.tt_orders").using("iceberg").createOrReplace()
snap1 = spark.sql("SELECT MAX(snapshot_id) FROM local.icedb.tt_orders.snapshots").collect()[0][0]

# Make changes
df.limit(2000).writeTo("local.icedb.tt_orders").append()
spark.sql("UPDATE local.icedb.tt_orders SET status='shipped' WHERE status='confirmed' LIMIT 1000")
spark.sql("DELETE FROM local.icedb.tt_orders WHERE status='cancelled' AND price < 25")

snap_latest = spark.sql("SELECT MAX(snapshot_id) FROM local.icedb.tt_orders.snapshots").collect()[0][0]
print(f"First snapshot : {snap1}")
print(f"Latest snapshot: {snap_latest}")
print(f"Current rows   : {spark.table('local.icedb.tt_orders').count():,}")

## Step 1 — VERSION AS OF (Snapshot ID)


In [ ]:
# Query at first snapshot
df_v1 = spark.read.option("snapshot-id", snap1).table("local.icedb.tt_orders")
print(f"At snapshot {snap1}: {df_v1.count():,} rows")
df_v1.groupBy("status").count().orderBy("status").show()

print(f"Current state : {spark.table('local.icedb.tt_orders').count():,} rows")
spark.table("local.icedb.tt_orders").groupBy("status").count().orderBy("status").show()

## Step 2 — TIMESTAMP AS OF


In [ ]:
import datetime as dt

# Get commit timestamps
snaps = spark.sql("""
    SELECT snapshot_id, committed_at, operation,
           summary['total-records'] AS total
    FROM local.icedb.tt_orders.snapshots
    ORDER BY committed_at
""").collect()

print("Timestamp-based time travel:")
for s in snaps:
    ts_str = s["committed_at"].strftime("%Y-%m-%d %H:%M:%S") if s["committed_at"] else None
    if ts_str:
        df_ts = spark.read.option("as-of-timestamp",
                                  str(int(s["committed_at"].timestamp()*1000))) \
                     .table("local.icedb.tt_orders")
        print(f"  {ts_str}: {df_ts.count():,} rows  ({s['operation']})")

## Step 3 — Tags: Named Checkpoints


In [ ]:
# Tag the current state as a named reference
spark.sql("""
    ALTER TABLE local.icedb.tt_orders
    CREATE TAG end_of_q1_2024 RETAIN 30 DAYS
""")

spark.sql("""
    ALTER TABLE local.icedb.tt_orders
    CREATE TAG baseline RETAIN 90 DAYS
""")

print("Named tags:")
spark.sql("SELECT name, type, snapshot_id FROM local.icedb.tt_orders.refs").show(truncate=False)

# Read via tag name
df_tagged = spark.read.option("snapshot-id",
    spark.sql("SELECT snapshot_id FROM local.icedb.tt_orders.refs WHERE name='baseline'")
         .collect()[0][0]
).table("local.icedb.tt_orders")
print(f"\nData at 'baseline' tag: {df_tagged.count():,} rows")
print("Tags make time travel readable: VERSION 'baseline' vs snapshot_id=8234567890...")

## Step 4 — Comparing Data Between Snapshots


In [ ]:
# Find what changed between two snapshots
snap_ids = [s["snapshot_id"] for s in snaps]
if len(snap_ids) >= 2:
    s_before = snap_ids[0]
    s_after  = snap_ids[-1]

    df_before = spark.read.option("snapshot-id", s_before).table("local.icedb.tt_orders")
    df_after  = spark.table("local.icedb.tt_orders")

    # Rows deleted
    deleted = df_before.join(df_after.select("order_id"), "order_id", "left_anti")
    print(f"Deleted between snap {s_before} and {s_after}: {deleted.count():,} rows")
    deleted.show(5)

    # Revenue change
    rev_before = df_before.agg(F.sum("revenue")).collect()[0][0]
    rev_after  = df_after.agg(F.sum("revenue")).collect()[0][0]
    print(f"\nRevenue change: {rev_before:,.2f} → {rev_after:,.2f}  (Δ {rev_after-rev_before:+,.2f})")

## Summary

```python
# By snapshot ID
spark.read.option("snapshot-id", snap_id).table("local.db.table")

# By timestamp (milliseconds)
spark.read.option("as-of-timestamp", str(ts_ms)).table("local.db.table")

# SQL syntax
spark.sql("SELECT * FROM local.db.table VERSION AS OF 12345")
spark.sql("SELECT * FROM local.db.table TIMESTAMP AS OF '2024-01-15 10:00:00'")

# Tags
spark.sql("ALTER TABLE local.db.table CREATE TAG my_tag RETAIN 30 DAYS")
# Read via tag:
snap_id = spark.sql("SELECT snapshot_id FROM local.db.table.refs WHERE name='my_tag'").collect()[0][0]
spark.read.option("snapshot-id", snap_id).table("local.db.table")
```

### Time travel window
Available as long as the data files exist.
Run `expire_snapshots` + `remove_orphan_files` to free space.
After cleanup, old snapshots are no longer queryable.
